In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, 256),
            nn.ReLU(),
        )

        self.policy_head = nn.Sequential(
            nn.Linear(256,64),
            nn.ReLU(),
            nn.Linear(64,act_dim),
        )
        self.value_head = nn.Sequential(
            nn.Linear(256,32),
            nn.ReLU(),
            nn.Linear(32,1),
        )

    def forward(self, x):
        x = self.shared(x)
        logits = self.policy_head(x)
        value = self.value_head(x)
        return logits, value

    def act(self, state):
        logits, value = self.forward(state)
        probs = torch.softmax(logits, dim=-1)
        dist = torch.distributions.Categorical(probs)

        action = dist.sample()
        log_prob = dist.log_prob(action)

        return action, log_prob, value

    def evaluate(self, states, actions):
        logits, values = self.forward(states)
        probs = torch.softmax(logits, dim=-1)
        dist = torch.distributions.Categorical(probs)

        log_probs = dist.log_prob(actions)
        entropy = dist.entropy()

        return log_probs, values.squeeze(-1), entropy

In [2]:
class RolloutBuffer:
    def __init__(self):
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.dones = []
        self.values = []

    def clear(self):
        self.__init__()

    def compute_returns_advantages(self, gamma=0.99, lam=0.95, num_envs=4):
        T = len(self.rewards)
        rollout_steps = T // num_envs

        # --- reshape into (num_envs, rollout_steps) ---
        rewards = torch.tensor(self.rewards, dtype=torch.float32).view(num_envs, rollout_steps)
        dones = torch.tensor(self.dones, dtype=torch.float32).view(num_envs, rollout_steps)
        values = torch.tensor(self.values, dtype=torch.float32).view(num_envs, rollout_steps)

        returns = torch.zeros_like(rewards)
        advantages = torch.zeros_like(rewards)

        # --- compute GAE per environment ---
        for env in range(num_envs):
            gae = 0
            next_value = 0  

            for t in reversed(range(rollout_steps)):
                delta = (
                    rewards[env, t]
                    + gamma * next_value * (1 - dones[env, t])
                    - values[env, t]
                )

                gae = delta + gamma * lam * (1 - dones[env, t]) * gae

                advantages[env, t] = gae
                returns[env, t] = gae + values[env, t]

                next_value = values[env, t]

        # --- flatten back to original shape (T,) ---
        returns = returns.view(-1)
        advantages = advantages.view(-1)

        return returns, advantages

In [3]:
import soccer_twos
from gym_unity.envs import ActionFlattener


def make_env(worker_id,mode=None):
    if mode is None:
        env = soccer_twos.make(worker = worker_id,)
    if mode == "random":
        env = soccer_twos.make(
            variation=soccer_twos.EnvType.team_vs_policy,
            single_player=True,
            worker = worker_id,
            flatten_branched=True,
        )
    if mode == "still":
        env = soccer_twos.make(
            opponent_policy=lambda *_: 0, 
            variation=soccer_twos.EnvType.team_vs_policy,
            single_player=True,
            worker = worker_id,
            flatten_branched=True,
        )
    # print(env.action_space.nvec)
    return env

In [4]:
import numpy as np
env = soccer_twos.make(
        variation=soccer_twos.EnvType.team_vs_policy,
        single_player=True,
        flatten_branched=True,
        worker = 2
    )
try: 
    print(env.action_space.n)
    print(env.observation_space.shape)
finally:env.close()
env = soccer_twos.make(
    )
try: 
    act = {i:np.array([0,0,0]) for i in range(4)}
    obs, reward, done, info = env.step(act)
    flattener = ActionFlattener(env.action_space.nvec)
    print(flattener.action_space.n)
    print(env.observation_space.shape)
    print(env.reset()[1].shape)
finally: env.close()

I0000 00:00:1776810362.062114  953922 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


...

[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0


27
(336,)
(336,)


In [5]:
env = make_env(3,"random")
try: 
    obs = env.reset()
    print("reset obs shape:", obs.shape)

    action = env.action_space.sample()
    print(f"action shape: {action}")
    obs, reward, done, info = env.step(0)

    print("step obs shape:", obs.shape)
    print("reward:", reward)
    print("done:", done)
    print(info)
finally:
    env.close()

I0000 00:00:1776810367.240561  953922 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0


reset obs shape: (336,)
action shape: 8
step obs shape: (336,)
reward: 0.0
done: False
{'player_info': {'position': array([-9.031397,  1.2     ], dtype=float32), 'rotation_y': 87.729774, 'velocity': array([0., 0.], dtype=float32)}, 'ball_info': {'position': array([1.0909986, 1.8254881], dtype=float32), 'velocity': array([0., 0.], dtype=float32)}}


In [41]:
from torch.utils.tensorboard import SummaryWriter
import time

log_dir = f"runs/soccer_ppo_{int(time.time())}"
# log_dir = "runs/soccer_ppo_1776750095"
writer = SummaryWriter(log_dir=log_dir)
%load_ext tensorboard


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [15]:
%tensorboard --logdir runs/

Reusing TensorBoard on port 6006 (pid 866175), started 11:31:09 ago. (Use '!kill 866175' to kill it.)

In [ ]:
def info_reward(info,team_signs):
    player_info = info["player_info"]
    ball_info = info["ball_info"]

    player_pos = player_info["position"]
    player_vel = player_info["velocity"]
    ball_pos = ball_info["position"]
    ball_vel = ball_info["velocity"]

    team_sign = team_signs
    def norm(v):
        return np.linalg.norm(v)

    def normalize(v):
        return v / (norm(v) + 1e-8)
    # --- compute ---
    to_ball = ball_pos - player_pos
    dist = norm(to_ball)
    to_ball_dir = normalize(to_ball)

    ball_speed = norm(ball_vel)
    ball_dir = normalize(ball_vel)

    goal_dir = np.array([team_sign, 0.0])
    own_goal_dir = -goal_dir

    is_touch = dist < 1.5

    forward_alignment = np.dot(to_ball, own_goal_dir)
    if forward_alignment >= 0:
        r_front_penalty = -0.06 * forward_alignment - 0.005
    else:
        r_front_penalty = 0.04

    if team_sign * ball_pos[0] <= 0:
        r_ball_position_penalty = 0.003 * team_sign * ball_pos[0]
    else:
        r_ball_position_penalty = 0.005 * team_sign * ball_pos[0]

    if r_front_penalty > 0:
        if ball_speed > 0.1: 
            if np.dot(ball_dir,goal_dir) >= 0: target_dir = -ball_dir
            else: target_dir = ball_dir
        else: 
            target_dir = own_goal_dir
        target_pos = ball_pos - own_goal_dir * 0.1
    else: target_pos = np.array([-14.5*team_sign,ball_pos[1]])

    to_target = target_pos - player_pos
    dist_target = norm(to_target)
    if r_front_penalty > 0:
        r_chase = 0.015 * np.dot(player_vel, normalize(to_target))
        r_dist = -0.035 * dist_target
    else:
        r_chase = 0.01 * np.dot(player_vel, normalize(to_target))
        r_dist = -0.03 * dist_target
        # if np.dot(to_ball_dir,normalize(player_vel)) <= np.cos(np.deg2rad(10)):

    speed = norm(player_vel)
    r_speed = 0.0
    if dist_target > 3.0:
        if r_front_penalty <= 0:
            if np.dot(normalize(player_vel),to_ball_dir) <= np.cos(np.deg2rad(10)):
                r_speed = 0.01 * speed
        else:
            if np.dot(normalize(player_vel),to_ball_dir) >= np.cos(np.deg2rad(10)):
                r_speed = 0.01 * speed

    cos_thres = np.cos(np.deg2rad(45))
    cos = np.dot(goal_dir,ball_dir)
    if cos > cos_thres and ball_speed > 0.1:
        cos = 1
    r_goal_dir = 0.25 * cos
    if ball_speed <= 0.1:
        r_goal_dir = -np.cos(np.deg2rad(75))

    r_btw_dir = 0.0
    ball_btw_agl = np.dot(normalize(np.array([16*team_sign,-3.8])-ball_pos),normalize(np.array([16*team_sign,3.8])-ball_pos))
    if cos > 0 and r_front_penalty > 0 and ball_speed > 0.1:
        if np.dot(ball_dir,normalize(np.array([16*team_sign,-3.8])-ball_pos))  >= ball_btw_agl:
            if np.dot(ball_dir,normalize(np.array([16*team_sign,3.8])-ball_pos))  >= ball_btw_agl:
                if ball_pos[0]*team_sign >= 0:
                    r_btw_dir += 0.015
                    r_dist *= 0.7
                    r_btw_dir += 0.01 * ball_speed
                    if not is_touch and ball_speed >= 0.3:
                        r_btw_dir += 0.05 * ball_speed
    r_block = 0
    if cos <= 0 and r_front_penalty >= 0:
        if np.dot(-to_ball_dir,ball_dir) >= np.cos(np.deg2rad(15)):
            r_block = 0.015
    if abs(cos) <= np.cos(np.deg2rad(80)) and ball_pos[0] * team_sign <= -2:
        r_block += 0.015
    r_impact = 0.0
    if is_touch:
        if r_front_penalty > 0:
            if cos >= 0:
                r_impact = 0.05 * cos + 0.025 * speed
                r_block *= 1.5
            else: 
                if np.dot(-to_ball_dir,ball_dir) >= np.cos(np.deg2rad(5)):
                    r_block += 0.04
        else:
            if cos < 0:
                r_impact = -0.5
    # --- total reward ---
    return r_chase, r_dist, r_goal_dir, r_impact, r_front_penalty, r_speed, r_ball_position_penalty, r_btw_dir, r_block

In [43]:
import torch.optim as optim
import numpy as np
# hyperparameters
NUM_ENVS = 1
ROLLOUT_STEPS = 512
EPOCHS = 10
BATCH_SIZE = 256
GAMMA = 0.99
LAMBDA = 0.95
CLIP_EPS = 0.2
LR = 3e-4

episode_rewards = []
current_rewards = [0 for _ in range(NUM_ENVS)]
global_step = 0


def collect_rollout(envs,model):
    global global_step
    buffer = RolloutBuffer()

    states = []
    for i,env in enumerate(envs):
        states.append(env.reset())
    states = np.array(states)
    team_signs = [None for _ in range(NUM_ENVS)]
    for _ in range(ROLLOUT_STEPS):
        state_tensor = torch.tensor(states, dtype=torch.float32)

        with torch.no_grad():
            actions, log_probs, values = model.act(state_tensor)

        next_states = []
        rewards = []
        dones = []

        for i, env in enumerate(envs):
                
            action = actions[i].item()
            obs, reward, done, info = env.step(action)
            if team_signs[i] is None:
                player_x = info["player_info"]["position"][0]
                team_signs[i] = +1 if player_x < 0 else -1
            r_chase, r_dist, r_goal_dir, r_impact, r_front_penalty, r_speed, r_ball_position_penalty, r_btw, r_block = info_reward(info,team_signs[i])
            r = reward + 1*(r_chase + r_dist + r_goal_dir + r_impact + r_front_penalty + r_speed + r_ball_position_penalty +r_btw + r_block)
            writer.add_scalar("reward/env", reward, global_step)
            writer.add_scalar("reward/block", r_block, global_step)
            writer.add_scalar("reward/btw", r_btw, global_step)
            writer.add_scalar("reward/chase", r_chase, global_step)
            writer.add_scalar("reward/dist", r_dist, global_step)
            writer.add_scalar("reward/goal_dir", r_goal_dir, global_step)
            writer.add_scalar("reward/impact", r_impact, global_step)
            writer.add_scalar("reward/front_penalty", r_front_penalty, global_step)
            writer.add_scalar("reward/pos_penalty", r_ball_position_penalty, global_step)
            writer.add_scalar("reward/speed", r_speed, global_step)
            writer.add_scalar("reward/total", r, global_step)
            if done:
                next_obs = env.reset()
                team_signs[i] = None

            else:
                next_obs = obs

            next_states.append(next_obs)
            rewards.append(r)
            dones.append(done)

        buffer.states.extend(state_tensor)
        buffer.actions.extend(actions)
        buffer.log_probs.extend(log_probs)
        buffer.rewards.extend(rewards)
        buffer.dones.extend(dones)
        buffer.values.extend(values.detach().view(-1).tolist())

        states = np.array(next_states)
        global_step += NUM_ENVS

    return buffer


def ppo_update(buffer,optimizer,model):
    returns, advantages = buffer.compute_returns_advantages(GAMMA, LAMBDA)

    states = torch.stack(buffer.states)
    actions = torch.stack(buffer.actions)
    old_log_probs = torch.stack(buffer.log_probs)

    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    for _ in range(EPOCHS):
        for i in range(0, len(states), BATCH_SIZE):
            s = states[i:i+BATCH_SIZE]
            a = actions[i:i+BATCH_SIZE]
            old_lp = old_log_probs[i:i+BATCH_SIZE]
            adv = advantages[i:i+BATCH_SIZE]
            ret = returns[i:i+BATCH_SIZE]

            log_probs, values, entropy = model.evaluate(s, a)

            ratio = torch.exp(log_probs - old_lp)
            surr1 = ratio * adv
            surr2 = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv

            actor_loss = -torch.min(surr1, surr2).mean()
            critic_loss = (ret - values).pow(2).mean()

            loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy.mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            writer.add_scalar("Loss/actor", actor_loss.item(), global_step)
            writer.add_scalar("Loss/critic", critic_loss.item(), global_step)
            writer.add_scalar("Loss/total", loss.item(), global_step)
            writer.add_scalar("Stats/entropy", entropy.mean().item(), global_step)


    


In [44]:
# training loop
# create environments
envs = [make_env(i+1,mode="still") for i in range(NUM_ENVS)]
obs_dim = envs[0].observation_space.shape[0]
act_dim = envs[0].action_space.n

model = ActorCritic(obs_dim, act_dim)
optimizer = optim.Adam(model.parameters(), lr=LR)
try:
    for episode in range(2000):
        buffer = collect_rollout(envs,model)
        ppo_update(buffer,optimizer,model)

        episode_reward = sum(buffer.rewards)
        writer.add_scalar("episode/total_reward", episode_reward, episode)
        if episode % 50 == 49:
            torch.save(model.state_dict(), "ppo_checkpoint_still_2000.pth")
            print(f"Saved checkpoint at episode {episode}")
finally:
    for env in envs:
        env.close()

I0000 00:00:1777009435.594955  866113 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0


Saved checkpoint at episode 49
Saved checkpoint at episode 99
Saved checkpoint at episode 149
Saved checkpoint at episode 199
Saved checkpoint at episode 249
Saved checkpoint at episode 299
Saved checkpoint at episode 349
Saved checkpoint at episode 399
Saved checkpoint at episode 449
Saved checkpoint at episode 499
Saved checkpoint at episode 549
Saved checkpoint at episode 599
Saved checkpoint at episode 649
Saved checkpoint at episode 699
Saved checkpoint at episode 749
Saved checkpoint at episode 799
Saved checkpoint at episode 849
Saved checkpoint at episode 899
Saved checkpoint at episode 949
Saved checkpoint at episode 999
Saved checkpoint at episode 1049
Saved checkpoint at episode 1099
Saved checkpoint at episode 1149
Saved checkpoint at episode 1199
Saved checkpoint at episode 1249
Saved checkpoint at episode 1299
Saved checkpoint at episode 1349
Saved checkpoint at episode 1399
Saved checkpoint at episode 1449
Saved checkpoint at episode 1499
Saved checkpoint at episode 1549

In [45]:
envs = [make_env(i+1,mode="random") for i in range(NUM_ENVS)]
try:
    for episode in range(2000,6000):
        buffer = collect_rollout(envs,model)
        ppo_update(buffer,optimizer,model)

        episode_reward = sum(buffer.rewards)
        writer.add_scalar("episode/total_reward", episode_reward, episode)
        if episode % 50 == 49:
            torch.save(model.state_dict(), "ppo_checkpoint_random_4000.pth")
            print(f"Saved checkpoint at episode {episode}")
finally:
    for env in envs:
        env.close()

I0000 00:00:1777014482.285436  866113 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0


Saved checkpoint at episode 2049
Saved checkpoint at episode 2099
Saved checkpoint at episode 2149
Saved checkpoint at episode 2199
Saved checkpoint at episode 2249
Saved checkpoint at episode 2299
Saved checkpoint at episode 2349
Saved checkpoint at episode 2399
Saved checkpoint at episode 2449
Saved checkpoint at episode 2499
Saved checkpoint at episode 2549
Saved checkpoint at episode 2599
Saved checkpoint at episode 2649
Saved checkpoint at episode 2699
Saved checkpoint at episode 2749
Saved checkpoint at episode 2799
Saved checkpoint at episode 2849
Saved checkpoint at episode 2899
Saved checkpoint at episode 2949
Saved checkpoint at episode 2999
Saved checkpoint at episode 3049
Saved checkpoint at episode 3099
Saved checkpoint at episode 3149
Saved checkpoint at episode 3199
Saved checkpoint at episode 3249
Saved checkpoint at episode 3299
Saved checkpoint at episode 3349
Saved checkpoint at episode 3399
Saved checkpoint at episode 3449
Saved checkpoint at episode 3499
Saved chec

In [46]:

# check_point_path = "ppo_checkpoint_random_4000.pth"
# model = ActorCritic(obs_dim, act_dim)
# optimizer = optim.Adam(model.parameters(), lr=LR)
# model.load_state_dict(torch.load(check_point_path))
envs = [make_env(i+1,mode="random") for i in range(NUM_ENVS)]
try:
    for episode in range(6000,10000):
        buffer = collect_rollout(envs,model)
        ppo_update(buffer,optimizer,model)

        episode_reward = sum(buffer.rewards)
        writer.add_scalar("episode/total_reward", episode_reward, episode)
        if episode % 50 == 49:
            torch.save(model.state_dict(), "ppo_checkpoint_random_8000.pth")
            print(f"Saved checkpoint at episode {episode}")
finally:
    for env in envs:
        env.close()
    

I0000 00:00:1777024690.188157  866113 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0


Saved checkpoint at episode 6049
Saved checkpoint at episode 6099
Saved checkpoint at episode 6149
Saved checkpoint at episode 6199
Saved checkpoint at episode 6249
Saved checkpoint at episode 6299
Saved checkpoint at episode 6349
Saved checkpoint at episode 6399
Saved checkpoint at episode 6449
Saved checkpoint at episode 6499
Saved checkpoint at episode 6549
Saved checkpoint at episode 6599
Saved checkpoint at episode 6649
Saved checkpoint at episode 6699
Saved checkpoint at episode 6749
Saved checkpoint at episode 6799
Saved checkpoint at episode 6849
Saved checkpoint at episode 6899
Saved checkpoint at episode 6949
Saved checkpoint at episode 6999
Saved checkpoint at episode 7049
Saved checkpoint at episode 7099
Saved checkpoint at episode 7149
Saved checkpoint at episode 7199
Saved checkpoint at episode 7249
Saved checkpoint at episode 7299
Saved checkpoint at episode 7349
Saved checkpoint at episode 7399
Saved checkpoint at episode 7449
Saved checkpoint at episode 7499
Saved chec

In [47]:

# check_point_path = "ppo_checkpoint_random_4000.pth"
# model = ActorCritic(obs_dim, act_dim)
# optimizer = optim.Adam(model.parameters(), lr=LR)
# model.load_state_dict(torch.load(check_point_path))
envs = [make_env(i+1,mode="random") for i in range(NUM_ENVS)]
try:
    for episode in range(10000,14000):
        buffer = collect_rollout(envs,model)
        ppo_update(buffer,optimizer,model)

        episode_reward = sum(buffer.rewards)
        writer.add_scalar("episode/total_reward", episode_reward, episode)
        if episode % 50 == 49:
            torch.save(model.state_dict(), "ppo_checkpoint_random_12000.pth")
            print(f"Saved checkpoint at episode {episode}")
finally:
    for env in envs:
        env.close()
    

I0000 00:00:1777034885.330526  866113 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0


Saved checkpoint at episode 10049
Saved checkpoint at episode 10099
Saved checkpoint at episode 10149
Saved checkpoint at episode 10199
Saved checkpoint at episode 10249
Saved checkpoint at episode 10299
Saved checkpoint at episode 10349
Saved checkpoint at episode 10399
Saved checkpoint at episode 10449
Saved checkpoint at episode 10499
Saved checkpoint at episode 10549
Saved checkpoint at episode 10599
Saved checkpoint at episode 10649
Saved checkpoint at episode 10699
Saved checkpoint at episode 10749
Saved checkpoint at episode 10799
Saved checkpoint at episode 10849
Saved checkpoint at episode 10899
Saved checkpoint at episode 10949
Saved checkpoint at episode 10999
Saved checkpoint at episode 11049
Saved checkpoint at episode 11099
Saved checkpoint at episode 11149
Saved checkpoint at episode 11199
Saved checkpoint at episode 11249
Saved checkpoint at episode 11299
Saved checkpoint at episode 11349
Saved checkpoint at episode 11399
Saved checkpoint at episode 11449
Saved checkpoi

In [ ]:
import random
def collect_rollout_self_play(envs,model,models):
    global global_step
    flattener = ActionFlattener(envs[0].action_space.nvec)
    buffer1 = RolloutBuffer()
    # buffer2 = RolloutBuffer()

    states = []
    for i,env in enumerate(envs):
        states.append(env.reset())
    
    team_signs = [None for _ in range(NUM_ENVS)]
    for _ in range(ROLLOUT_STEPS):
        s1 = []
        s2 = []
        s3 = []
        s4 = []
        for s in states:
            s1.append(s[0])
            s2.append(s[1])
            s3.append(s[2])
            s4.append(s[3])
        state_tensor1 = torch.tensor(s1, dtype=torch.float32)
        state_tensor2 = torch.tensor(s2, dtype=torch.float32)
        state_tensor3 = torch.tensor(s3, dtype=torch.float32)
        state_tensor4 = torch.tensor(s4, dtype=torch.float32)
        opp_model = random.choice(models)

        with torch.no_grad():
            actions1, log_probs1, values1 = model.act(state_tensor1)
            # actions2, log_probs2, values2 = model.act(state_tensor2)
            actions3, _, _ = opp_model.act(state_tensor3)
            # actions4, _, _ = opp_model.act(state_tensor4)

        next_states = []
        rewards1 = []
        dones1 = []
        rewards2 = []
        dones2 = []

        for i, env in enumerate(envs):
                
            action1 = actions1[i].item()
            # action2 = actions2[i].item()
            action3 = actions3[i].item()
            # action4 = actions4[i].item()
            action2 = env.action_space.sample()
            action4 = env.action_space.sample()
            action = {}
            action[0] = flattener.lookup_action(action1)
            action[1] = flattener.lookup_action(action2)
            action[2] = flattener.lookup_action(action3)
            action[3] = flattener.lookup_action(action4)
            obs, reward, done, info = env.step(action)
            # print(done)
            if team_signs[i] is None:
                player_x = info[0]["player_info"]["position"][0]
                team_signs[i] = +1 if player_x < 0 else -1
            r_chase1, r_dist1, r_goal_dir1, r_impact1, r_front_penalty1, r_speed1, r_ball_position_penalty1, r_btw1 = info_reward(info[0],team_signs[i])
            # r_chase2, r_dist2, r_goal_dir2, r_impact2, r_front_penalty2, r_speed2, r_ball_position_penalty2, r_btw2 = info_reward(info[1],team_signs[i])
            r1 = reward[0] + r_chase1 + r_dist1 + r_goal_dir1 + r_impact1 + r_front_penalty1 + r_speed1 + r_ball_position_penalty1 +r_btw1
            # r2 = reward[1] + r_chase2 + r_dist2 + r_goal_dir2 + r_impact2 + r_front_penalty2 + r_speed2 + r_ball_position_penalty2 +r_btw2
            writer.add_scalar("reward/env", reward[0], global_step)
            writer.add_scalar("reward/btw", r_btw1, global_step)
            writer.add_scalar("reward/chase", r_chase1, global_step)
            writer.add_scalar("reward/dist", r_dist1, global_step)
            writer.add_scalar("reward/goal_dir", r_goal_dir1, global_step)
            writer.add_scalar("reward/impact", r_impact1, global_step)
            writer.add_scalar("reward/front_penalty", r_front_penalty1, global_step)
            writer.add_scalar("reward/pos_penalty", r_ball_position_penalty1, global_step)
            writer.add_scalar("reward/speed", r_speed1, global_step)
            writer.add_scalar("reward/total", r1, global_step)
            if done:
                next_obs = env.reset()
                team_signs[i] = None

            else:
                next_obs = obs

            next_states.append(next_obs)
            rewards1.append(r1)
            # rewards2.append(r2)
            dones1.append(done["__all__"])
            # dones2.append(done["__all__"])

        buffer1.states.extend(state_tensor1)
        buffer1.actions.extend(actions1)
        buffer1.log_probs.extend(log_probs1)
        buffer1.rewards.extend(rewards1)
        buffer1.dones.extend(dones1)
        buffer1.values.extend(values1.detach().view(-1).tolist())
        # buffer2.states.extend(state_tensor2)
        # buffer2.actions.extend(actions2)
        # buffer2.log_probs.extend(log_probs2)
        # buffer2.rewards.extend(rewards2)
        # buffer2.dones.extend(dones2)
        # buffer2.values.extend(values2.detach().view(-1).tolist())

        states = next_states
        global_step += NUM_ENVS

    return buffer1#,buffer2

In [ ]:
envs = [make_env(i+1,mode=None) for i in range(NUM_ENVS)]
# check_point_path = "ppo_checkpoint_baseline_5000.pth"
# model = ActorCritic(obs_dim, act_dim)
# optimizer = optim.Adam(model.parameters(), lr=LR)
# model.load_state_dict(torch.load(check_point_path))
try:
    for episode in range(10000,15000):
        buffer1 = collect_rollout_self_play(envs,model)
        ppo_update(buffer1,optimizer,model)

        episode_reward = sum(buffer1.rewards) 
        writer.add_scalar("episode/total_reward", episode_reward, episode)
        print(episode)
        if episode % 50 == 49:
            torch.save(model.state_dict(), "ppo_checkpoint_selfplay_5000.pth")
            print(f"Saved checkpoint at episode {episode}")
finally:
    for env in envs:
        env.close()
    # writer.close()

In [14]:
from ceia_baseline_agent.agent_ray import RayAgent
def collect_rollout_two(envs,model):
    global global_step
    flattener = ActionFlattener(envs[0].action_space.nvec)
    buffer1 = RolloutBuffer()
    buffer2 = RolloutBuffer()
    baseline = RayAgent(envs[0])

    states = []
    for i,env in enumerate(envs):
        states.append(env.reset())
    
    team_signs = [None for _ in range(NUM_ENVS)]
    for _ in range(ROLLOUT_STEPS):
        s1 = []
        s2 = []
        for s in states:
            s1.append(s[0])
            s2.append(s[1])
        state_tensor1 = torch.tensor(s1, dtype=torch.float32)
        state_tensor2 = torch.tensor(s2, dtype=torch.float32)

        with torch.no_grad():
            actions1, log_probs1, values1 = model.act(state_tensor1)
            actions2, log_probs2, values2 = model.act(state_tensor2)

        next_states = []
        rewards1 = []
        dones1 = []
        rewards2 = []
        dones2 = []

        for i, env in enumerate(envs):
                
            action1 = actions1[i].item()
            action2 = actions2[i].item()
            action = baseline.act({2:states[i][2],3:states[i][3]})
            action[0] = flattener.lookup_action(action1)
            action[1] = flattener.lookup_action(action2)
            obs, reward, done, info = env.step(action)
            # print(done)
            if team_signs[i] is None:
                player_x = info[0]["player_info"]["position"][0]
                team_signs[i] = +1 if player_x < 0 else -1
            r_chase1, r_dist1, r_goal_dir1, r_impact1, r_front_penalty1, r_speed1, r_ball_position_penalty1, r_btw1, r_block1 = info_reward(info[0],team_signs[i])
            r_chase2, r_dist2, r_goal_dir2, r_impact2, r_front_penalty2, r_speed2, r_ball_position_penalty2, r_btw2, r_block2 = info_reward(info[1],team_signs[i])
            r1 = reward[0] + r_chase1 + r_dist1 + r_goal_dir1 + r_impact1 + r_front_penalty1 + r_speed1 + r_ball_position_penalty1 +r_btw1+r_block1
            r2 = reward[1] + r_chase2 + r_dist2 + r_goal_dir2 + r_impact2 + r_front_penalty2 + r_speed2 + r_ball_position_penalty2 +r_btw2+r_block2
            writer.add_scalar("reward/env", reward[0], global_step)
            writer.add_scalar("reward/block", r_block1, global_step)
            writer.add_scalar("reward/btw", r_btw1, global_step)
            writer.add_scalar("reward/chase", r_chase1, global_step)
            writer.add_scalar("reward/dist", r_dist1, global_step)
            writer.add_scalar("reward/goal_dir", r_goal_dir1, global_step)
            writer.add_scalar("reward/impact", r_impact1, global_step)
            writer.add_scalar("reward/front_penalty", r_front_penalty1, global_step)
            writer.add_scalar("reward/pos_penalty", r_ball_position_penalty1, global_step)
            writer.add_scalar("reward/speed", r_speed1, global_step)
            writer.add_scalar("reward/total", r1, global_step)
            if done:
                next_obs = env.reset()
                team_signs[i] = None

            else:
                next_obs = obs

            next_states.append(next_obs)
            rewards1.append(r1)
            rewards2.append(r2)
            dones1.append(done["__all__"])
            dones2.append(done["__all__"])

        buffer1.states.extend(state_tensor1)
        buffer1.actions.extend(actions1)
        buffer1.log_probs.extend(log_probs1)
        buffer1.rewards.extend(rewards1)
        buffer1.dones.extend(dones1)
        buffer1.values.extend(values1.detach().view(-1).tolist())
        buffer2.states.extend(state_tensor2)
        buffer2.actions.extend(actions2)
        buffer2.log_probs.extend(log_probs2)
        buffer2.rewards.extend(rewards2)
        buffer2.dones.extend(dones2)
        buffer2.values.extend(values2.detach().view(-1).tolist())

        states = next_states
        global_step += NUM_ENVS

    return buffer1,buffer2
def ppo_update_2(buffer1,buffer2,optimizer,model):
    returns1, advantages1 = buffer1.compute_returns_advantages(GAMMA, LAMBDA)
    returns2, advantages2 = buffer2.compute_returns_advantages(GAMMA, LAMBDA)

    states1 = torch.stack(buffer1.states)
    actions1 = torch.stack(buffer1.actions)
    logp1 = torch.stack(buffer1.log_probs)

    states2 = torch.stack(buffer2.states)
    actions2 = torch.stack(buffer2.actions)
    logp2 = torch.stack(buffer2.log_probs)

    states = torch.cat([states1, states2], dim=0)
    actions = torch.cat([actions1, actions2], dim=0)
    old_log_probs = torch.cat([logp1, logp2], dim=0)
    returns = torch.cat([returns1, returns2], dim=0)
    advantages = torch.cat([advantages1, advantages2], dim=0)

    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    N = states.size(0)

    for _ in range(EPOCHS):
        idx = torch.randperm(N)
        for i in range(0, N, BATCH_SIZE):
            batch_idx = idx[i:i+BATCH_SIZE]
            s = states[batch_idx]
            a = actions[batch_idx]
            old_lp = old_log_probs[batch_idx]
            adv = advantages[batch_idx]
            ret = returns[batch_idx]

            log_probs, values, entropy = model.evaluate(s, a)

            ratio = torch.exp(log_probs - old_lp)
            surr1 = ratio * adv
            surr2 = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv

            actor_loss = -torch.min(surr1, surr2).mean()
            critic_loss = (ret - values).pow(2).mean()

            loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy.mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            writer.add_scalar("Loss/actor", actor_loss.item(), global_step)
            writer.add_scalar("Loss/critic", critic_loss.item(), global_step)
            writer.add_scalar("Loss/total", loss.item(), global_step)
            writer.add_scalar("Stats/entropy", entropy.mean().item(), global_step)

/home/clark/anaconda3/envs/soccertwos/lib/python3.8/site-packages/ray/autoscaler/_private/cli_logger.py:57: FutureWarning: Not all Ray CLI dependencies were found. In Ray 1.4+, the Ray CLI, autoscaler, and dashboard will only be usable via `pip install 'ray[default]'`. Please update your install command.
  warnings.warn(


In [ ]:
envs = [make_env(i+1,mode=None) for i in range(NUM_ENVS)]
# model1 = ActorCritic(obs_dim, act_dim)
# model1.load_state_dict(model.state_dict())
# optimizer1 = optim.Adam(model1.parameters(), lr=LR)
try:
    for episode in range(10000,15000):
        buffer1,buffer2 = collect_rollout_two(envs,model)
        ppo_update_2(buffer1,buffer2,optimizer,model)

        episode_reward = (sum(buffer1.rewards) + sum(buffer1.rewards))/2
        writer.add_scalar("episode/total_reward", episode_reward, episode)
        print(episode)
        if episode % 50 == 49:
            torch.save(model.state_dict(), "ppo_checkpoint_baseline_5000.pth")
            print(f"Saved checkpoint at episode {episode}")
finally:
    for env in envs:
        env.close()
    try:
        import ray
        ray.shutdown()
    except:
        pass
    # writer.close()

In [ ]:
envs = [make_env(i+1,mode=None) for i in range(NUM_ENVS)]
check_point_path = "ppo_checkpoint_baseline_5000.pth"
model = ActorCritic(obs_dim, act_dim)
optimizer = optim.Adam(model.parameters(), lr=LR)
model.load_state_dict(torch.load(check_point_path))
try:
    for episode in range(15000,19000):
        buffer1,buffer2 = collect_rollout_two(envs,model)
        ppo_update_2(buffer1,buffer2,optimizer,model)

        episode_reward = (sum(buffer1.rewards) + sum(buffer1.rewards))/2
        writer.add_scalar("episode/total_reward", episode_reward, episode)
        print(episode)
        if episode % 50 == 49:
            torch.save(model.state_dict(), "ppo_checkpoint_baseline_9000.pth")
            print(f"Saved checkpoint at episode {episode}")
finally:
    for env in envs:
        env.close()
    try:
        import ray
        ray.shutdown()
    except:
        pass
    # writer.close()

In [16]:
try:
    import ray
    ray.shutdown()
except:
    pass

In [ ]:
import random
def collect_rollout_self_play(envs,model,models):
    global global_step
    flattener = ActionFlattener(envs[0].action_space.nvec)
    buffer1 = RolloutBuffer()
    # buffer2 = RolloutBuffer()

    states = []
    for i,env in enumerate(envs):
        states.append(env.reset())
    
    team_signs = [None for _ in range(NUM_ENVS)]
    for _ in range(ROLLOUT_STEPS):
        s1 = []
        s2 = []
        s3 = []
        s4 = []
        for s in states:
            s1.append(s[0])
            s2.append(s[1])
            s3.append(s[2])
            s4.append(s[3])
        state_tensor1 = torch.tensor(s1, dtype=torch.float32)
        state_tensor2 = torch.tensor(s2, dtype=torch.float32)
        state_tensor3 = torch.tensor(s3, dtype=torch.float32)
        state_tensor4 = torch.tensor(s4, dtype=torch.float32)
        opp_model = random.choice(models)

        with torch.no_grad():
            actions1, log_probs1, values1 = model.act(state_tensor1)
            actions2, log_probs2, values2 = model.act(state_tensor2)
            actions3, _, _ = opp_model.act(state_tensor3)
            actions4, _, _ = opp_model.act(state_tensor4)

        next_states = []
        rewards1 = []
        dones1 = []
        rewards2 = []
        dones2 = []

        for i, env in enumerate(envs):
                
            action1 = actions1[i].item()
            action2 = actions2[i].item()
            action3 = actions3[i].item()
            action4 = actions4[i].item()
            action2 = env.action_space.sample()
            action4 = env.action_space.sample()
            sction = {}
            action[0] = flattener.lookup_action(action1)
            action[1] = flattener.lookup_action(action2)
            action[2] = flattener.lookup_action(action3)
            action[3] = flattener.lookup_action(action4)
            obs, reward, done, info = env.step(action)
            # print(done)
            if team_signs[i] is None:
                player_x = info[0]["player_info"]["position"][0]
                team_signs[i] = +1 if player_x < 0 else -1
            r_chase1, r_dist1, r_goal_dir1, r_impact1, r_front_penalty1, r_speed1, r_ball_position_penalty1, r_btw1 = info_reward(info[0],team_signs[i])
            r_chase2, r_dist2, r_goal_dir2, r_impact2, r_front_penalty2, r_speed2, r_ball_position_penalty2, r_btw2 = info_reward(info[1],team_signs[i])
            r1 = reward[0] + r_chase1 + r_dist1 + r_goal_dir1 + r_impact1 + r_front_penalty1 + r_speed1 + r_ball_position_penalty1 +r_btw1
            r2 = reward[1] + r_chase2 + r_dist2 + r_goal_dir2 + r_impact2 + r_front_penalty2 + r_speed2 + r_ball_position_penalty2 +r_btw2
            writer.add_scalar("reward/env", reward[0], global_step)
            writer.add_scalar("reward/btw", r_btw1, global_step)
            writer.add_scalar("reward/chase", r_chase1, global_step)
            writer.add_scalar("reward/dist", r_dist1, global_step)
            writer.add_scalar("reward/goal_dir", r_goal_dir1, global_step)
            writer.add_scalar("reward/impact", r_impact1, global_step)
            writer.add_scalar("reward/front_penalty", r_front_penalty1, global_step)
            writer.add_scalar("reward/pos_penalty", r_ball_position_penalty1, global_step)
            writer.add_scalar("reward/speed", r_speed1, global_step)
            writer.add_scalar("reward/total", r1, global_step)
            if done:
                next_obs = env.reset()
                team_signs[i] = None

            else:
                next_obs = obs

            next_states.append(next_obs)
            rewards1.append(r1)
            rewards2.append(r2)
            dones1.append(done["__all__"])
            dones2.append(done["__all__"])

        buffer1.states.extend(state_tensor1)
        buffer1.actions.extend(actions1)
        buffer1.log_probs.extend(log_probs1)
        buffer1.rewards.extend(rewards1)
        buffer1.dones.extend(dones1)
        buffer1.values.extend(values1.detach().view(-1).tolist())
        # buffer2.states.extend(state_tensor2)
        # buffer2.actions.extend(actions2)
        # buffer2.log_probs.extend(log_probs2)
        # buffer2.rewards.extend(rewards2)
        # buffer2.dones.extend(dones2)
        # buffer2.values.extend(values2.detach().view(-1).tolist())

        states = next_states
        global_step += NUM_ENVS

    return buffer1#,buffer2

In [14]:
envs = [make_env(i+1,mode="still") for i in range(NUM_ENVS)]
obs_dim = envs[0].observation_space.shape[0]
act_dim = envs[0].action_space.n

for env in envs:
        env.close()

I0000 00:00:1776810911.770861  953922 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0
